In [ ]:
import cupy as cp
import cv2
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

import matplotlib.transforms as transforms
from cupyx.scipy.ndimage import gaussian_filter
from ipywidgets import (
    IntSlider,
    fixed,
    interact,
)
from matplotlib.patches import Rectangle

from config import IMAGE_DATA_DIR
from kuwahara_filters import (
    anisotropic_kuwahara,
    classic_kuwahara,
    generalized_kuwahara,
)

In [ ]:
def load_and_resize(image_path, max_side=800):
    img = cv2.imread(image_path, cv2.IMREAD_COLOR_RGB)
    h, w = img.shape[:2]

    if max(h, w) <= max_side:
        return img

    scale = max_side / float(max(h, w))
    new_w = int(w * scale)
    new_h = int(h * scale)

    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

In [ ]:
def add_gaussian_noise(image, sigma):
    if sigma == 0:
        return image.copy()

    noise = np.random.normal(0, sigma, image.shape)
    noisy_image = image.astype(np.float32) + noise

    return np.clip(noisy_image, 0, 255).astype(np.uint8)

In [ ]:
def update_kuwahara(image, noise_sigma, kuwahara_algorithm, **kuwahara_params):
    noisy_image = add_gaussian_noise(image, sigma=noise_sigma)
    denoised_image = kuwahara_algorithm(noisy_image, **kuwahara_params)

    fig, axes = plt.subplots(1, 3, figsize=(15, 7))

    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(noisy_image)
    axes[1].set_title(f"Noisy (sigma={noise_sigma})")
    axes[1].axis("off")

    axes[2].imshow(denoised_image)
    params = ",".join(
        "=".join(map(str, pair)) for pair in kuwahara_params.items()
    )
    axes[2].set_title(f"Filtered ({params})")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
image = load_and_resize(IMAGE_DATA_DIR / "landscape.png", max_side=600)

interact(
    update_kuwahara,
    image=fixed(image),
    kuwahara_algorithm=fixed(anisotropic_kuwahara),
    noise_sigma=IntSlider(min=0, max=100, step=5, value=10),
    radius=IntSlider(min=3, max=10, step=1, value=5),
)

In [ ]:
noisy = add_gaussian_noise(image, sigma=10)
general_filtered = generalized_kuwahara(noisy, radius=5)
anisotropic_filtered = anisotropic_kuwahara(noisy, radius=5)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0][0].imshow(image)
axes[0][0].set_title("Original")

axes[0][1].imshow(noisy)
axes[0][1].set_title("Noisy")

axes[1][0].imshow(general_filtered)
axes[1][0].set_title("Generalized Kuwahara")

axes[1][1].imshow(anisotropic_filtered)
axes[1][1].set_title("Anisotropic Kuwahara")

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
y, x = 315, 200
h, w = 75, 75

general_with_box = general_filtered.copy()
cv2.rectangle(general_with_box, (x, y), (x + w, y + h), 255, 2)

anisotropic_with_box = anisotropic_filtered.copy()
cv2.rectangle(anisotropic_with_box, (x, y), (x + w, y + h), 255, 2)

crop_general = general_filtered[y : y + h, x : x + w]
crop_anisotropic = anisotropic_filtered[y : y + h, x : x + w]

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

axes[0][0].imshow(general_with_box)
axes[0][0].set_title("General filter", fontsize=14)

axes[0][1].imshow(anisotropic_with_box)
axes[0][1].set_title("Anisotropic filter", fontsize=14)

axes[1][0].imshow(crop_general)
axes[1][0].set_title("General filter (Zoom)")

axes[1][1].imshow(crop_anisotropic)
axes[1][1].set_title("Anisotropic filter (Zoom)")

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
image = load_and_resize(IMAGE_DATA_DIR / "mountain_iic.jpg", max_side=150)

upscaled_image = cv2.resize(
    image, dsize=None, fx=4, fy=4, interpolation=cv2.INTER_CUBIC
)
anisotropic_filtered = anisotropic_kuwahara(upscaled_image, radius=9)

y, x = 200, 490
h, w = 100, 100

upscaled_with_box = upscaled_image.copy()
cv2.rectangle(upscaled_with_box, (x, y), (x + w, y + h), 255, 2)

anisotropic_with_box = anisotropic_filtered.copy()
cv2.rectangle(anisotropic_with_box, (x, y), (x + w, y + h), 255, 2)

crop_upscaled = upscaled_image[y : y + h, x : x + w]
crop_anisotropic = anisotropic_filtered[y : y + h, x : x + w]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

axes[0, 0].imshow(upscaled_with_box)
axes[0, 0].set_title("Upscaled image", fontsize=14)

axes[0, 1].imshow(anisotropic_with_box)
axes[0, 1].set_title("Anisotropic filter", fontsize=14)

axes[1, 0].imshow(crop_upscaled)
axes[1, 0].set_title("Upscaled image (Zoom)")

axes[1, 1].imshow(crop_anisotropic)
axes[1, 1].set_title("Anisotropic filter (Zoom)")

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
image = load_and_resize(IMAGE_DATA_DIR / "wolf.jpg", max_side=700)

classic_filtered = classic_kuwahara(image, radius=5)
general_filtered = generalized_kuwahara(image, radius=5)
anisotropic_filtered = anisotropic_kuwahara(image, radius=5)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0][0].imshow(image)
axes[0][0].set_title("Original")

axes[0][1].imshow(classic_filtered)
axes[0][1].set_title("Classic Kuwahara")

axes[1][0].imshow(general_filtered)
axes[1][0].set_title("General Kuwahara")

axes[1][1].imshow(anisotropic_filtered)
axes[1][1].set_title("Anisotropic Kuwahara")

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
image = load_and_resize(IMAGE_DATA_DIR / "parrot.png", max_side=600)
image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

image_float = cp.asarray(image, dtype=cp.float32)
Ix = gaussian_filter(
    image_float, sigma=1.0, order=(0, 1), mode="nearest", axes=(0, 1)
)
Iy = gaussian_filter(
    image_float, sigma=1.0, order=(1, 0), mode="nearest", axes=(0, 1)
)

E = gaussian_filter(Ix * Ix, sigma=2.0, mode="nearest")
F = gaussian_filter(Ix * Iy, sigma=2.0, mode="nearest")
G = gaussian_filter(Iy * Iy, sigma=2.0, mode="nearest")

discriminant_root = cp.sqrt(cp.clip((E - G) ** 2 + 4 * F**2, 0, None))
lambda1 = (E + G + discriminant_root) / 2.0

phi_1 = cp.arctan2(Ix, -Iy)
phi_2 = cp.arctan2(-F, lambda1 - E)

cx, cy = 290, 230
crop_size = 15

image_with_box = cv2.cvtColor(image.copy(), cv2.COLOR_GRAY2RGB)
cv2.rectangle(
    image_with_box,
    (cx - crop_size, cy - crop_size),
    (cx + crop_size, cy + crop_size),
    255,
    2,
)

img_crop = image[
    cy - crop_size : cy + crop_size + 1, cx - crop_size : cx + crop_size + 1
]
phi_1_crop = cp.asnumpy(
    phi_1[
        cy - crop_size : cy + crop_size + 1, cx - crop_size : cx + crop_size + 1
    ]
)
phi_2_crop = cp.asnumpy(
    phi_2[
        cy - crop_size : cy + crop_size + 1, cx - crop_size : cx + crop_size + 1
    ]
)

fig, axes = plt.subplots(1, 3, figsize=(12, 6))

axes[0].imshow(image_with_box)
axes[0].set_title("Original image")

axes[1].imshow(img_crop, cmap="gray")
axes[1].set_title("Tangent field (no smoothing)")
axes[1].set_xticks(np.arange(-0.5, 19, 1), minor=True)
axes[1].set_yticks(np.arange(-0.5, 19, 1), minor=True)
axes[1].grid(
    which="minor", color="black", linestyle="-", linewidth=0.5, alpha=0.2
)
axes[1].tick_params(which="minor", size=0)

rect_w = 0.8
rect_h = 0.05

for i in range(crop_size * 2 + 1):
    for j in range(crop_size * 2 + 1):
        angle_deg = np.degrees(phi_1_crop[i, j].item())

        rect = Rectangle(
            (j - rect_w / 2, i - rect_h / 2),
            width=rect_w,
            height=rect_h,
            color="blue",
            alpha=0.8,
        )

        t = (
            transforms.Affine2D().rotate_deg_around(j, i, angle_deg)
            + axes[1].transData
        )
        rect.set_transform(t)

        axes[1].add_patch(rect)

axes[2].imshow(img_crop, cmap="gray")
axes[2].set_title("Tangent field (smoothed)")
axes[2].set_xticks(np.arange(-0.5, 19, 1), minor=True)
axes[2].set_yticks(np.arange(-0.5, 19, 1), minor=True)
axes[2].grid(
    which="minor", color="black", linestyle="-", linewidth=0.5, alpha=0.2
)
axes[2].tick_params(which="minor", size=0)

for i in range(crop_size * 2 + 1):
    for j in range(crop_size * 2 + 1):
        angle_deg = np.degrees(phi_2_crop[i, j].item())

        rect = Rectangle(
            (j - rect_w / 2, i - rect_h / 2),
            width=rect_w,
            height=rect_h,
            color="blue",
            alpha=0.8,
        )

        t = (
            transforms.Affine2D().rotate_deg_around(j, i, angle_deg)
            + axes[2].transData
        )
        rect.set_transform(t)

        axes[2].add_patch(rect)

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()